# Your First Object-Oriented Agent

In this short tutorial we'll explore the core principles that make NOOA special. We'll start with a small toy example: a `BaristaAgent` whose job is to recommend drinks to sleepy customers. Along the way, we'll see that a NOOA agent is **just a Python object**. We give it new tools by adding methods. We spin up multiple agents by instantiating the same class N times. We strongly type its outputs like any regular Python function.

We won't try to sell you on a whole new paradigm. What you'll actually walk away with is plain Python, with a tiny sprinkle of magic.

## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below uses a placeholder API key. Replace `"your-api-key"` with a real key to actually run the LLM calls. Any [LiteLLM-supported](https://docs.litellm.ai/) model works.

## Setup

In [3]:
from nooa.unifiedllm.registry import get_llm_client

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")

## A Barista Agent

Let's start with our first agent. In NOOA, you define an agent by subclassing the `Agent` class. Here is a complete, working agent. Read every line — there isn't much to read.

In [7]:
from nooa import Agent

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""  # this becomes the agent's system prompt

    async def recommend_drink(self, customer_request: str) -> str:
        """Recommend a single drink to the customer based on what they just told you.
        Be warm and concise — one sentence is plenty."""
        ...

Now let's instantiate and call it. Notice that we `await` the method — generation methods are async.

In [8]:
barista = BaristaAgent()
result = await barista.recommend_drink("Hi, I feel so sleepy.")
print(result)

I'd love to make you a double shot espresso or a strong cold brew to help perk you up—both will give you that energy boost you need!


That's it — a friendly recommendation, ready to serve. So what just happened?

Behind the scenes, NOOA does a bit of work to keep the interface this Pythonic. A few things worth noticing:

- the **class docstring** became the agent's system prompt
- the **method docstring** became the task description
- the **ellipsis (`...`)** is how you tell NOOA "this method is *agentic* — hand it off to the LLM instead of running it as normal Python"

More concretely, NOOA quietly assembles a prompt that looks roughly like:

```
<system prompt>
You are a friendly barista at a small neighborhood cafe.

<available methods>
recommend_drink(customer_request: str) -> str
Recommend a single drink to the customer based on what they just told you. Be warm and concise — one sentence is plenty.
</available methods>
```

Plus a handful of other bits we'll unpack later. Let's call the agent once more, just for fun:

In [ ]:
result = await barista.recommend_drink("I would love something that works with a cornetto.")
print(result)

> 📝 **Takeaway:** the agent is an object.

## Adding Tools

Our friendly barista just offered a strong caffeine kick — but it's 9pm as we write this, and a double espresso is definitely not the move. How do we teach the agent to respect a "no caffeine after 4pm" policy?

In some agent frameworks you'd have to register a tool, describe it in a JSON schema, and wire it into the runtime. In NOOA, you just **add a method to the class**. Anything the agent can see on itself, it can call as a tool.

Let's add our first tool to `BaristaAgent`.

In [ ]:
from datetime import datetime
from nooa import Agent

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def is_only_decaf_hour(self) -> bool:
        """Return True if we should only be serving decaf right now. After 4pm we go decaf-only so our customers can still sleep tonight."""
        return datetime.now().hour >= 16

    async def recommend_drink(self, customer_request: str) -> str:
        """Recommend a single drink to the customer based on what they told you."""
        ...

barista = BaristaAgent()
await barista.recommend_drink("Hi, I feel so sleepy.")

"I'd recommend a decaf cappuccino! It's smooth and comforting, even without the caffeine boost."

> 📝 **Takeaway:** in NOOA, ordinary Python methods and agentic methods live side by side on the same class. The agent freely calls the deterministic ones as tools — no registration, no schema, no glue code.

## Strong Typing

What if our cafe only offers a fixed menu and we want the agent to *only* recommend drinks we actually sell?

Most agent frameworks deal with a single data type: text. Text gets passed to tools, text gets exchanged between agents, and text gets returned as output — and then you painstakingly parse it back into JSON, hoping the model got the shape right (which is not always the case, even with the best models). NOOA takes a different approach: because our agent lives inside a Python program, we can strongly type everything.

Let's put a real type on the return value of `recommend_drink`:

In [ ]:
from enum import Enum
from nooa import Agent


class Drink(Enum):
    ESPRESSO = "espresso"
    CAPPUCCINO = "cappuccino"
    FLAT_WHITE = "flat white"

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def is_only_decaf_hour(self) -> bool:
        """Return True if we should only be serving decaf right now. After 4pm we go decaf-only so our customers can still sleep tonight."""
        return datetime.now().hour >= 16

    async def recommend_drink(self, customer_request: str) -> tuple[str, Drink]:
        """Pick the single best drink for the customer from the menu, based on what they told you."""
        ...

barista = BaristaAgent()
reason, drink = await barista.recommend_drink("Hi, I feel so sleepy.")
print("Barista: ", reason)
print("Recommended drink: ", drink)

Perfect! An espresso is just what you need - it's our strongest caffeine hit and will help wake you right up!
Drink.ESPRESSO


This isn't just prompt engineering — NOOA enforces the return type at runtime. If the LLM tries to return something that isn't a valid `Drink`, the framework retries until it does. You get real Python objects back, not strings you have to guess-parse.

> 📝 **Takeaway:** strong typing all the way through.

## Give Your Agent Some State

Our cafe has a finite stash of coffee beans, and every drink burns through a few. Since our agent is *just a Python object*, giving it state is as easy as adding a field in `__init__`:

In [18]:
from nooa import Agent

class BaristaAgent(Agent, llm=model):
    """You are a friendly barista at a small neighborhood cafe."""

    def __init__(self, coffee_beans: int) -> None:
        super().__init__()
        self.coffee_beans = coffee_beans

    def is_only_decaf_hour(self) -> bool:
        """Return True if we should only be serving decaf right now. After 4pm we go decaf-only so our customers can still sleep tonight."""
        return datetime.now().hour >= 16

    async def recommend_drink(self, customer_request: str) -> tuple[str, Drink]:
        """Pick the single best drink for the customer from the menu, based on what they told you."""
        self.coffee_beans -= 1
        ...

In [19]:
barista = BaristaAgent(coffee_beans=2)
for _ in range(3):
    barista_answer, drink = await barista.recommend_drink("Something to keep me going, please.")
    print(f"Answer: {barista_answer}\nServed: {drink}\nBeans left: {barista.coffee_beans}")

Answer: I'd recommend an espresso! It's our strongest option and will definitely keep you going.
Served: Drink.ESPRESSO
Beans left: 1
Answer: I'd recommend an espresso! It's our strongest option and will definitely keep you going.
Served: Drink.ESPRESSO
Beans left: 0
Answer: I'm so sorry, but we've run out of coffee beans! I won't be able to make you anything right now. Can I interest you in coming back tomorrow when we restock?
Served: Drink.ESPRESSO
Beans left: -1


In [20]:
from nooa import print_prompt
await print_prompt(barista.recommend_drink, customer_request="I could use a pick-me-up")

=== SYSTEM PROMPT  [BaristaAgent] ===

<system_prompt expr="self._system_prompt()">
You are BaristaAgent, a Python agent working in an interactive session.

## Context blocks
Your prompt is organized in XML context blocks: `<name>CONTENT</name>`.
Blocks produced by `self.context.set_dynamic()` carry an `expr="..."` attribute whose value is the Python expression re-evaluated each turn.
Event history: system entries in `<sys tag="N">`; reference via `self.events["N"]`.

## Truncation
- A bare Python literal (`[1, 2, 3]`, `{1: 2}`, `'hello'`) is always complete.
- Truncated values use a `type(len=N, ...)` (or `type(repr_len=N, ...)`) marker:
    list(len=100, [:5]=[...], [-5:]=[...])
    tuple(len=100, [:5]=(...), [-5:]=(...))
    dict(len=100, items={...})
    set(len=100, items={...})
    str(len=100000, [:250]='...', [-250:]='...')
    ndarray(repr_len=233, [:100]='...', [-100:]='...')
- Structured instances (dataclasses, Pydantic, custom classes) render as `ClassName(field=value, ...)

> 📝 **Takeaway:** the agent "sees" everything, including access to itself.

## Peeking Inside the Context

As we mentioned, when you write an agentic method, NOOA's harness builds the real prompt for you and makes sure the LLM sees exactly what it needs — and nothing more. Curious what the agent actually sees? Two helpers to keep in your back pocket:

- `doc(self)` — the auto-generated API view of your agent (methods, docstrings, types). This is the same view the LLM gets.
- `print_prompt(agent.method, ...)` — the fully rendered prompt for a specific method call, template variables and all.

## What Is Happening Under the Hood?

NOOA doesn't hide the prompt from you. `print_prompt` renders exactly what would be sent to the LLM for a given method call — the system prompt, the agent introspection block, and the task. Let's take a look:

In [18]:
await nooa.print_prompt(agent.recommend_drink, mood="stressed and running late")

=== SYSTEM PROMPT  [BaristaAgent] ===

<system_prompt expr="self._system_prompt()">
You are BaristaAgent, a Python agent working in an interactive session.

## Context blocks
Your prompt is organized in XML context blocks: `<name>CONTENT</name>`.
Blocks produced by `self.context.set_dynamic()` carry an `expr="..."` attribute whose value is the Python expression re-evaluated each turn.
Event history: system entries in `<sys tag="N">`; reference via `self.events["N"]`.

## Truncation
- A bare Python literal (`[1, 2, 3]`, `{1: 2}`, `'hello'`) is always complete.
- Truncated values use a `type(len=N, ...)` (or `type(repr_len=N, ...)`) marker:
    list(len=100, [:5]=[...], [-5:]=[...])
    tuple(len=100, [:5]=(...), [-5:]=(...))
    dict(len=100, items={...})
    set(len=100, items={...})
    str(len=100000, [:250]='...', [-250:]='...')
    ndarray(repr_len=233, [:100]='...', [-100:]='...')
- Structured instances (dataclasses, Pydantic, custom classes) render as `ClassName(field=value, ...)

Look through the output. There is no hidden state — this is exactly what the LLM sees. A few blocks worth naming:

- **`<system_prompt>`** — opens with your class docstring. This is the persona the model wears for every method on the class.
- **`<strategy_prompt>`** — a compact rulebook explaining how to act each turn: what tools exist (`execute_python`, `return_result`), when to use which, and how to finish a run. This is what teaches the LLM to "inhabit" the framework, and it's the same for every agent.
- **`<execution_context>`** — the imports, types, and helpers that will be in scope when the LLM writes code. Anything you import at module level shows up here.
- **`<self>`** — auto-generated documentation of the agent's public methods and fields, rendered from `doc(type(self))`. This is how the LLM discovers what the agent can do — no separate tool registry.
- **Task prompt** — your method docstring, with parameters like `mood` bound to the values you passed in. Rendered live at call time.

That's the whole prompt. No hidden system messages, no template files, no per-tool JSON schemas glued on the side. The reason the "rulebook" stays short is that the framework is plain Python, and the model already knows Python.

> **Tip:** `print_prompt` shows the *outgoing* prompt only. To watch a whole run unfold — the LLM's response, generated code, tool calls, retries, validation — the framework ships a live **trace viewer** (`nemo start-dev`, served on `localhost:5001`). Any agent you create will stream into it automatically once it's running.

### Where the Magic Happens

Two things to sit with:

- **You didn't register `is_only_decaf_hour` anywhere.** No `@tool`, no JSON schema, no `tools=[...]` list. The framework rendered `doc(self)` into the system prompt (that `<self>` block you saw earlier), and the LLM discovered the method the same way you would when reading someone's code.
- **The generation method used the helper without being told to.** Nothing in `recommend_drink`'s docstring mentions `is_only_decaf_hour`. The LLM spotted a method that looked relevant, called it, and folded the result into its recommendation. That is the whole "tools are just methods" idea in one page.

### No Tool Registration

> **Why this is different:** every other agent framework asks you to register tools — usually a name, a JSON schema for arguments, a description string, sometimes a return schema. Here you just define a method. Type hints become the schema. The docstring becomes the description. Python does all the work the framework would otherwise be doing on top of it.

The corollary: renaming a method renames the tool. Adding a parameter changes the tool signature. Deleting a method removes the tool. Refactor as you would refactor any Python class.

## Recap

The five things the barista taught us:

- **Ellipsis `...` marks a generation method.** No decorator, no separate registry. If the body is `...`, the LLM implements it.
- **The class docstring is the system prompt; the method docstring is the task.** Rewriting a prompt means editing a docstring.
- **`{self.attr}` in docstrings is live.** Change the attribute, and the next call sees the new value.
- **Every non-hidden method on `self` is a tool.** No `@tool` decorator, no JSON schema, no registration step.
- **The return type annotation is the output contract.** Pydantic models are validated and retried automatically.

## Exercises

Try these in a fresh cell. Solutions are one small edit each.

1. **Daily special.** Add a `daily_special: str` field to `BaristaAgent.__init__` and reference it in the `recommend_drink` docstring with `{self.daily_special}`. Confirm the LLM starts pushing that drink.
2. **Refund method.** Add a `refund(self, drink: str) -> str` generation method that produces the barista's grudging apology and refund line.
3. **Time of day.** Add a `time_of_day: Literal["morning", "afternoon", "closing"]` field to the return type and rerun. Confirm it appears in the output — no other code change required.
4. **Bean anxiety.** Modify the `recommend_drink` docstring so the barista is documented as noticeably snarkier when `self.coffee_beans < 5`. Set `barista.coffee_beans = 2` and observe the snark level rise.